# 04 — Continue the SFT adapter with GRPO

For each task, GRPO samples four structured completions, scores them, and reinforces
completions that are better relative to the group. The reward stays intentionally
simple:

\[
R = F\times(0.10 + 0.90J)
\]

`F` is the exact format gate. `J` is `gpt-5.6-luna`'s normalized four-dimension
scientific judgment.

## Important execution note

The semantic reward is live API work because the policy creates new completions
during training. The cache makes reruns resumable. This notebook intentionally
requires one training process; a distributed production run should use a
concurrency-safe reward service.

## Configuration

Every model, reward coefficient, sampling setting, optimizer parameter, checkpoint
setting, path, and Hub repository used below is an explicit variable.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from accelerate import PartialState
from datasets import load_dataset, load_from_disk
from IPython.display import JSON, Markdown, display
from peft import PeftModel
from trl import GRPOConfig, GRPOTrainer

from science_course.devices import clear_device_cache, detect_runtime
from science_course.hub import require_hf_namespace
from science_course.judge import (
    DEFAULT_JUDGE_MODEL,
    configure_scientific_design_judge,
    scientific_design_reward,
)
from science_course.modeling import (
    generate_text,
    load_causal_lm,
    load_tokenizer,
    render_prompt,
)
from science_course.teacher import require_openai_key
from science_course.versions import require_training_stack

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Runtime, input sources, and local artifacts
ENABLE_MPS_FALLBACK = True
TOKENIZERS_PARALLELISM = False
if ENABLE_MPS_FALLBACK:
    os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = str(TOKENIZERS_PARALLELISM).lower()
MODEL_ID = "google/gemma-4-E4B-it"
GRPO_DATA_SOURCE_MODE = "hub"  # "hub" or "local"
SFT_ADAPTER_SOURCE_MODE = "hub"  # "hub" or "local"
LOCAL_SFT_ADAPTER = ROOT / "artifacts" / "gemma4-scientific-design-sft"
LOCAL_GRPO_DATA = ROOT / "data" / "processed" / "scientific_design_grpo"
GRPO_ADAPTER = ROOT / "artifacts" / "gemma4-scientific-design-grpo"
RESUME_FROM_CHECKPOINT = None

# Exact Luna reward
JUDGE_MODEL = DEFAULT_JUDGE_MODEL
JUDGE_CACHE = ROOT / "results" / "scientific_design_judge_cache.jsonl"
FORMAT_BASE_REWARD = 0.10
SEMANTIC_REWARD_WEIGHT = 0.90
REWARD_FUNCTION_WEIGHTS = [1.0]
OPENAI_MAX_RETRIES = 3
OPENAI_TIMEOUT_SECONDS = 120.0

# GRPO optimization and sampling
NUM_TRAIN_EPOCHS = 1
MAX_STEPS = -1
LEARNING_RATE = 5e-6
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
NUM_GENERATIONS = 4
NUM_GENERATIONS_EVAL = 4
MAX_COMPLETION_LENGTH = 512
TEMPERATURE = 0.8
TOP_P = 0.95
TOP_K = 0
BETA = 0.0
SCALE_REWARDS = "group"
MULTI_OBJECTIVE_AGGREGATION = "sum_then_normalize"
LOSS_TYPE = "dapo"
GRADIENT_CHECKPOINTING = True
GRADIENT_CHECKPOINTING_USE_REENTRANT = False
OPTIMIZER = "adamw_torch"
WARMUP_STEPS = 5
# Full GRPO evaluation is expensive: 75 tasks × 4 rollouts = 300
# candidate responses plus Luna grading. With one epoch, "epoch" runs it once.
# For periodic evaluation instead, use "steps" and set EVAL_STEPS (for example 250).
EVAL_STRATEGY = "epoch"
EVAL_STEPS = None
SAVE_STRATEGY = "steps"
SAVE_STEPS = 25
SAVE_TOTAL_LIMIT = None
LOGGING_STEPS = 1
LOGGING_FIRST_STEP = True
LOG_COMPLETIONS = True
NUM_COMPLETIONS_TO_PRINT = 4
REPORT_TO = "none"
USE_VLLM = False
RANDOM_SEED = 17
PIN_MEMORY_ON_CUDA_ONLY = True

# Before/after inference comparison
MAX_NEW_TOKENS = 512
INFERENCE_DO_SAMPLE = False
INFERENCE_TEMPERATURE = 1.0
INFERENCE_TOP_P = 1.0

# Hugging Face publication
DATASET_HF_REPO = "lamm-mit/scientific-sft-grpo-data"
DATASET_CONFIG_NAME = "scientific_design_grpo"
SFT_HF_REPO = "lamm-mit/scientific-sft-grpo-design-sft"
GRPO_HF_REPO = "lamm-mit/scientific-sft-grpo-design-grpo"
PUSH_TO_HUB = True
HUB_STRATEGY = "all_checkpoints"
HUB_PRIVATE_REPO = False
HUB_ALWAYS_PUSH = True
HF_TOKEN = None
# HF_TOKEN = os.environ["HF_TOKEN"]  # Optional; prefer `hf auth login`.

require_openai_key()
versions = require_training_stack()
runtime = detect_runtime()
if GRPO_DATA_SOURCE_MODE not in {"hub", "local"}:
    raise ValueError("GRPO_DATA_SOURCE_MODE must be 'hub' or 'local'.")
if SFT_ADAPTER_SOURCE_MODE not in {"hub", "local"}:
    raise ValueError("SFT_ADAPTER_SOURCE_MODE must be 'hub' or 'local'.")
SFT_ADAPTER_SOURCE = (
    SFT_HF_REPO
    if SFT_ADAPTER_SOURCE_MODE == "hub"
    else LOCAL_SFT_ADAPTER
)
DATALOADER_PIN_MEMORY = (
    runtime.backend == "cuda" if PIN_MEMORY_ON_CUDA_ONLY else True
)
if JUDGE_MODEL != "gpt-5.6-luna":
    raise RuntimeError("The configured semantic judge must be gpt-5.6-luna.")
if abs(FORMAT_BASE_REWARD + SEMANTIC_REWARD_WEIGHT - 1.0) > 1e-9:
    raise ValueError("Reward coefficients must sum to one.")
if EVAL_BATCH_SIZE % NUM_GENERATIONS_EVAL != 0:
    raise ValueError(
        "EVAL_BATCH_SIZE must be divisible by NUM_GENERATIONS_EVAL."
    )
if PartialState().num_processes != 1:
    raise RuntimeError("This API-judged teaching run requires WORLD_SIZE=1.")
configure_scientific_design_judge(
    model=JUDGE_MODEL,
    cache_path=JUDGE_CACHE,
    format_base_reward=FORMAT_BASE_REWARD,
    semantic_reward_weight=SEMANTIC_REWARD_WEIGHT,
    api_max_retries=OPENAI_MAX_RETRIES,
    api_timeout_seconds=OPENAI_TIMEOUT_SECONDS,
)
if PUSH_TO_HUB:
    require_hf_namespace(GRPO_HF_REPO, token=HF_TOKEN)
display(
    JSON(
        {
            "runtime": runtime.as_dict(),
            "versions": versions,
            "model": MODEL_ID,
            "grpo_dataset_source_mode": GRPO_DATA_SOURCE_MODE,
            "sft_adapter_source_mode": SFT_ADAPTER_SOURCE_MODE,
            "sft_adapter": str(SFT_ADAPTER_SOURCE),
            "judge_model": JUDGE_MODEL,
            "reward_formula": "F * (0.10 + 0.90 * J)",
            "hub_model": GRPO_HF_REPO,
        }
    )
)

## 1. Load the task-only GRPO data and trainable SFT adapter

The policy receives only `prompt`. TRL passes the hidden rubric columns to the
reward function.

In [ ]:
if GRPO_DATA_SOURCE_MODE == "hub":
    grpo = load_dataset(
        DATASET_HF_REPO,
        DATASET_CONFIG_NAME,
        token=HF_TOKEN,
    )
else:
    if not LOCAL_GRPO_DATA.exists():
        raise RuntimeError(
            "LOCAL_GRPO_DATA does not exist. Run notebook 02 or use "
            "GRPO_DATA_SOURCE_MODE='hub'."
        )
    grpo = load_from_disk(LOCAL_GRPO_DATA)
if (
    SFT_ADAPTER_SOURCE_MODE == "local"
    and not LOCAL_SFT_ADAPTER.exists()
):
    raise RuntimeError(
        "LOCAL_SFT_ADAPTER does not exist. Run notebook 03 or use "
        "SFT_ADAPTER_SOURCE_MODE='hub'."
    )
if len(grpo["train"]) == 0 or len(grpo["validation"]) == 0:
    raise RuntimeError("GRPO train and validation splits must be non-empty.")

tokenizer = load_tokenizer(MODEL_ID, token=HF_TOKEN)
clear_device_cache(runtime)
base_model = load_causal_lm(MODEL_ID, runtime, token=HF_TOKEN)
model = PeftModel.from_pretrained(
    base_model,
    SFT_ADAPTER_SOURCE,
    is_trainable=True,
    token=HF_TOKEN,
)
model.print_trainable_parameters()
print(grpo)

## 2. Snapshot the SFT policy on one unseen test task

This qualitative before/after comparison is useful for teaching but is not a
substitute for aggregate expert evaluation.

In [ ]:
held_out = grpo["test"][0]
held_out_prompt = render_prompt(tokenizer, held_out["prompt"])
before_grpo = generate_text(
    model,
    tokenizer,
    held_out_prompt,
    runtime,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=INFERENCE_DO_SAMPLE,
    temperature=INFERENCE_TEMPERATURE,
    top_p=INFERENCE_TOP_P,
)
display(Markdown("### SFT policy\n```text\n" + before_grpo + "\n```"))

## 3. Configure the exact combined reward

For each completion:

1. `F=1` only when all four non-empty sections appear in exact order; otherwise
   `F=0` and the reward is zero without calling Luna.
2. Luna assigns integer 0–4 scores for brainstorm, principles, synthesis, and
   answer.
3. `J=(B+P+S+A)/16`.
4. `R=F×(0.10+0.90J)`.

GRPO then normalizes these raw rewards within each four-completion group.

In [ ]:
grpo_args = GRPOConfig(
    output_dir=str(GRPO_ADAPTER),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_generations=NUM_GENERATIONS,
    num_generations_eval=NUM_GENERATIONS_EVAL,
    max_completion_length=MAX_COMPLETION_LENGTH,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    beta=BETA,
    reward_weights=REWARD_FUNCTION_WEIGHTS,
    scale_rewards=SCALE_REWARDS,
    multi_objective_aggregation=MULTI_OBJECTIVE_AGGREGATION,
    loss_type=LOSS_TYPE,
    remove_unused_columns=False,
    chat_template_kwargs={"enable_thinking": False},
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    gradient_checkpointing_kwargs={
        "use_reentrant": GRADIENT_CHECKPOINTING_USE_REENTRANT
    },
    optim=OPTIMIZER,
    warmup_steps=WARMUP_STEPS,
    eval_strategy=EVAL_STRATEGY,
    eval_steps=EVAL_STEPS,
    save_strategy=SAVE_STRATEGY,
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    logging_steps=LOGGING_STEPS,
    logging_first_step=LOGGING_FIRST_STEP,
    log_completions=LOG_COMPLETIONS,
    num_completions_to_print=NUM_COMPLETIONS_TO_PRINT,
    report_to=REPORT_TO,
    dataloader_pin_memory=DATALOADER_PIN_MEMORY,
    push_to_hub=PUSH_TO_HUB,
    hub_model_id=GRPO_HF_REPO,
    hub_strategy=HUB_STRATEGY,
    hub_private_repo=HUB_PRIVATE_REPO,
    hub_token=HF_TOKEN,
    hub_always_push=HUB_ALWAYS_PUSH,
    bf16=runtime.trainer_bf16,
    fp16=runtime.trainer_fp16,
    use_cpu=runtime.use_cpu,
    use_vllm=USE_VLLM,
    seed=RANDOM_SEED,
)
trainer = GRPOTrainer(
    model=model,
    args=grpo_args,
    reward_funcs=[scientific_design_reward],
    train_dataset=grpo["train"],
    eval_dataset=grpo["validation"],
    processing_class=tokenizer,
)

## 4. Train, evaluate, checkpoint, and publish

The Luna cache is append-only and content-addressed, so completed judgments are
reused after interruption. Every saved trainer checkpoint is published.

With the default `EVAL_STRATEGY="epoch"` and one training epoch, held-out
evaluation runs once at the end. It generates four fresh rollouts for each of the
75 validation tasks and applies the same reward function—up to 300 Luna judgments.
These `eval_*` rewards are diagnostics and do not update the policy. The
unprefixed training reward is the signal used by GRPO.

In [ ]:
train_result = trainer.train(
    resume_from_checkpoint=RESUME_FROM_CHECKPOINT
)
trainer.save_model(GRPO_ADAPTER)
tokenizer.save_pretrained(GRPO_ADAPTER)
if PUSH_TO_HUB:
    hub_result = trainer.push_to_hub(
        commit_message="Complete scientific design GRPO training"
    )
    print(f"Published final adapter and all checkpoints: {hub_result}")
display(JSON(train_result.metrics))

In [ ]:
history = pd.DataFrame(trainer.state.log_history)
training_reward_columns = [
    column
    for column in history.columns
    if column == "reward"
    or (column.startswith("rewards/") and column.endswith("/mean"))
]
evaluation_reward_columns = [
    column
    for column in history.columns
    if column == "eval_reward"
    or (
        column.startswith("eval_rewards/")
        and column.endswith("/mean")
    )
]
reward_columns = training_reward_columns + evaluation_reward_columns
if reward_columns:
    history[["step", *reward_columns]].dropna(
        how="all",
        subset=reward_columns,
    ).plot(
        x="step",
        y=reward_columns,
        figsize=(12, 5),
        marker="o",
    )
    plt.title("Training reward and held-out evaluation reward")
    plt.ylabel("reward")
    plt.tight_layout()
    plt.show()
else:
    print("Reward columns available:", sorted(history.columns))

## 5. Compare the same unseen task

Inspect whether the GRPO policy develops more distinct candidates, states the
relevant constraints, synthesizes rather than lists, and provides a defensible
final answer.

In [ ]:
after_grpo = generate_text(
    trainer.model,
    tokenizer,
    held_out_prompt,
    runtime,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=INFERENCE_DO_SAMPLE,
    temperature=INFERENCE_TEMPERATURE,
    top_p=INFERENCE_TOP_P,
)
display(Markdown("### Before GRPO\n```text\n" + before_grpo + "\n```"))
display(Markdown("### After GRPO\n```text\n" + after_grpo + "\n```"))
display(Markdown("### Hidden instructor rubric"))
display(
    JSON(
        {
            "required_constraints": held_out["required_constraints"],
            "evaluation_criteria": held_out["evaluation_criteria"],
            "acceptable_alternatives": held_out["acceptable_alternatives"],
            "failure_modes": held_out["failure_modes"],
        }
    )
)

## 6. Fresh-kernel inference from Hugging Face

The preceding comparison intentionally uses the in-memory trainer. This cell is
independent of that state: it can be run after reopening the notebook or in a
fresh kernel. By default it loads the final adapter from the root of the GRPO Hub
repository. Set `INFERENCE_ADAPTER_SUBFOLDER` to a published `checkpoint-*`
directory, or set `INFERENCE_ADAPTER_REVISION` to a branch, tag, or commit.

Inference requires Hugging Face access to gated Gemma weights but does **not**
require an OpenAI key or a Luna call.

In [ ]:
import gc
import os

from IPython.display import Markdown, display
from peft import PeftModel

from science_course.data import task_prompt
from science_course.devices import clear_device_cache, detect_runtime
from science_course.modeling import (
    generate_text,
    load_causal_lm,
    load_tokenizer,
    render_prompt,
)

# All fresh-kernel inference settings are explicit here.
INFERENCE_BASE_MODEL_ID = "google/gemma-4-E4B-it"
INFERENCE_ADAPTER_REPO = "lamm-mit/scientific-sft-grpo-design-grpo"
INFERENCE_ADAPTER_REVISION = None  # Optional branch, tag, or commit hash.
INFERENCE_ADAPTER_SUBFOLDER = None  # Example: "checkpoint-100".
INFERENCE_MAX_NEW_TOKENS = 512
INFERENCE_DO_SAMPLE = False
INFERENCE_TEMPERATURE = 1.0
INFERENCE_TOP_P = 1.0
INFERENCE_HF_TOKEN = None
# INFERENCE_HF_TOKEN = os.environ["HF_TOKEN"]  # Optional; cached login is preferred.

INFERENCE_TASK = (
    "Design a self-healing hydrogel for repeated deformation in water. "
    "The material may use reversible physical interactions, but recovery must "
    "not require external heating. Develop several mechanistic strategies, "
    "identify the governing design principles, synthesize the strongest design, "
    "and give a final recommendation."
)

# Release any prior trainer/model objects if this cell follows training.
for object_name in ("trainer", "model", "base_model"):
    globals().pop(object_name, None)
gc.collect()

inference_runtime = detect_runtime()
clear_device_cache(inference_runtime)
inference_tokenizer = load_tokenizer(
    INFERENCE_BASE_MODEL_ID,
    token=INFERENCE_HF_TOKEN,
)
inference_base_model = load_causal_lm(
    INFERENCE_BASE_MODEL_ID,
    inference_runtime,
    token=INFERENCE_HF_TOKEN,
)
adapter_load_kwargs = {
    "is_trainable": False,
    "token": INFERENCE_HF_TOKEN,
}
if INFERENCE_ADAPTER_REVISION is not None:
    adapter_load_kwargs["revision"] = INFERENCE_ADAPTER_REVISION
if INFERENCE_ADAPTER_SUBFOLDER is not None:
    adapter_load_kwargs["subfolder"] = INFERENCE_ADAPTER_SUBFOLDER
inference_model = PeftModel.from_pretrained(
    inference_base_model,
    INFERENCE_ADAPTER_REPO,
    **adapter_load_kwargs,
)

inference_prompt = render_prompt(
    inference_tokenizer,
    task_prompt(INFERENCE_TASK),
)
inference_response = generate_text(
    inference_model,
    inference_tokenizer,
    inference_prompt,
    inference_runtime,
    max_new_tokens=INFERENCE_MAX_NEW_TOKENS,
    do_sample=INFERENCE_DO_SAMPLE,
    temperature=INFERENCE_TEMPERATURE,
    top_p=INFERENCE_TOP_P,
)
display(
    Markdown(
        "### Reloaded GRPO adapter response\n"
        f"Repository: `{INFERENCE_ADAPTER_REPO}`  \n"
        f"Subfolder: `{INFERENCE_ADAPTER_SUBFOLDER or 'final adapter'}`\n\n"
        f"```text\n{inference_response}\n```"
    )
)

## Result and research caveat

The final adapter is in `artifacts/gemma4-scientific-design-grpo/` and on the
configured Hub repository with all saved checkpoints. For research claims, add
expert grading, inter-rater agreement, multiple seeds, reward ablations,
contamination checks, and domain-specific held-out evaluations. Luna provides
scalable supervision—not scientific ground truth.